In [23]:
from load_params import load_and_rotate
import pandas as pd
import sys
sys.path.append('..') 
import style
# Concatenate math and gsm outputs for theta, a, b
theta, a, b = load_and_rotate('../result/mirt-fitting/mirt_model_k2_legalbench.pt', rotation=None)
theta_k6, a_k6, b_k6 = load_and_rotate('../result/mirt-fitting/mirt_model_k6_legalbench.pt', rotation=None)

resmat = pd.read_pickle('../data/resmat.pkl')
conv = ['legalbench']

conv_mask = resmat.loc[:, resmat.columns.get_level_values("scenario").isin(conv)]

conv_questions = conv_mask.columns.get_level_values('input.text').tolist()


theta_df = pd.DataFrame(theta, columns=[f'Factor_{i+1}' for i in range(theta.shape[1])], index=resmat.index)
theta_df_k6 = pd.DataFrame(theta_k6, columns=[f'Factor_{i+1}' for i in range(theta_k6.shape[1])], index=resmat.index)
a_df = pd.DataFrame(a, columns=[f'Factor_{i+1}' for i in range(a.shape[1])], index=conv_questions)
a_df_k6 = pd.DataFrame(a_k6, columns=[f'Factor_{i+1}' for i in range(a_k6.shape[1])], index=conv_questions)

models_answered = conv_mask.dropna(how='all').index.tolist()
theta_df = theta_df.loc[models_answered]
theta_df_k6 = theta_df_k6.loc[models_answered]

b_df = pd.DataFrame(b, columns=['Difficulty'], index=conv_questions)
b_df_k6 = pd.DataFrame(b_k6, columns=['Difficulty'], index=conv_questions)

--- Loading from Cache ---
Loaded cached theta shape: (183, 2)
Loaded cached a shape: (1997, 2)
Loaded cached b shape: (1997,)

--- Loading from Cache ---
Loaded cached theta shape: (183, 6)
Loaded cached a shape: (1997, 6)
Loaded cached b shape: (1997,)



In [ ]:
from evaluation_model import *

# -------------------
# Run all models
# -------------------
# Load response matrix (already aligned)
Y = resmat.loc[models_answered, resmat.columns.get_level_values('input.text').isin(conv_questions)].values
n_obs = Y[~np.isnan(Y)].size

# === Rasch (1PL) ===
theta_uni = pd.read_csv('../result/irt-fitting/calibration_result_theta_legalbench.csv').values.ravel()
b_uni = pd.read_csv('../result/irt-fitting/calibration_result_z_legalbench.csv').values.ravel()
ll_rasch = loglik_rasch(Y, theta_uni, b_uni)
fit_rasch = aic_bic_from_loglik(ll_rasch, num_params=len(b_uni), n_obs=n_obs)

print("=== Rasch (1PL) ===")
print(fit_rasch)

# === MIRT (2D) ===
theta_mirt = theta_df.values      # N x 2
a_mirt = a_df.values              # J x 2
b_mirt = b_df.values.ravel()      # J
ll_mirt = loglik_mirt(Y, theta_mirt, a_mirt, b_mirt)
fit_mirt = aic_bic_from_loglik(ll_mirt, num_params=a_mirt.size + b_mirt.size, n_obs=n_obs)

print("\n=== MIRT (2D) ===")
print(fit_mirt)

# === MIRT (6D) ===
theta_mirt_k6 = theta_df_k6.values    # N x 6
a_mirt_k6 = a_df_k6.values            # J x 6
b_mirt_k6 = b_df_k6.values.ravel()    # J
ll_mirt_k6 = loglik_mirt(Y, theta_mirt_k6, a_mirt_k6, b_mirt_k6)
fit_mirt_k6 = aic_bic_from_loglik(ll_mirt_k6, num_params=a_mirt_k6.size + b_mirt_k6.size, n_obs=n_obs)

print("\n=== MIRT (6D) ===")
print(fit_mirt_k6)

# -------------------
# Likelihood Ratio Tests
# -------------------
print("\n=== Likelihood Ratio Tests ===")
delta_2d, df_2d, pval_2d = likelihood_ratio_test(
    ll_rasch, ll_mirt,
    df_small=len(b_uni), df_large=a_mirt.size + b_mirt.size
)
print(f"Rasch vs MIRT-2D: Δ(-2LL)={delta_2d:.2f}, df={df_2d}, p={pval_2d:.2e}")

delta_6d, df_6d, pval_6d = likelihood_ratio_test(
    ll_mirt, ll_mirt_k6,
    df_small=a_mirt.size + b_mirt.size, df_large=a_mirt_k6.size + b_mirt_k6.size
)
print(f"MIRT-2D vs MIRT-6D: Δ(-2LL)={delta_6d:.2f}, df={df_6d}, p={pval_6d:.2e}")

# -------------------
# Diagnostics (per-response + pseudo-R²)
# -------------------
print("\n=== Per-response Log-likelihood ===")
print(f"Rasch (1PL): {per_response_ll(ll_rasch, Y):.4f}")
print(f"MIRT (2D):  {per_response_ll(ll_mirt, Y):.4f}")
print(f"MIRT (6D):  {per_response_ll(ll_mirt_k6, Y):.4f}")

print("\n=== Pseudo-R² (McFadden, relative to Rasch) ===")
print(f"MIRT (2D): {pseudo_r2(ll_mirt, ll_rasch):.4f}")
print(f"MIRT (6D): {pseudo_r2(ll_mirt_k6, ll_rasch):.4f}")


=== Rasch (1PL) ===
{'loglik': np.float64(-269041.13204248017), '-2LL': np.float64(538082.2640849603), 'AIC': np.float64(542076.2640849603), 'BIC': np.float64(562255.7670919506)}

=== MIRT (2D) ===
{'loglik': np.float64(-68236.00585524479), '-2LL': np.float64(136472.01171048958), 'AIC': np.float64(148454.01171048958), 'BIC': np.float64(208992.52073146036)}

=== MIRT (6D) ===
{'loglik': np.float64(-55265.3071224266), '-2LL': np.float64(110530.6142448532), 'AIC': np.float64(138488.6142448532), 'BIC': np.float64(279745.135293785)}

=== Likelihood Ratio Tests ===
Rasch vs MIRT-2D: Δ(-2LL)=401610.25, df=3994, p=0.00e+00
MIRT-2D vs MIRT-6D: Δ(-2LL)=25941.40, df=7988, p=0.00e+00

=== Per-response Log-likelihood ===
Rasch (1PL): -1.4884
MIRT (2D):  -0.3775
MIRT (6D):  -0.3057

=== Pseudo-R² (McFadden, relative to Rasch) ===
MIRT (2D): 0.7464
MIRT (6D): 0.7946
